# 正解 — Stretch 04 — 分布と外れ値

`Body_Mass`の平均値、標準偏差、種ごとの分位点、外れ値件数を全体と種ごとで表示してください。


## 準備

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../../../data/penguins.parquet")
x = df["Body_Mass"].dropna().to_numpy()  # NumPy配列

平均と標準偏差を計算してください。


In [ ]:
m = x.mean()
s = x.std(ddof=1)
print(f"{m=}, {s=}")

種ごとの`Body_Mass`の分位点（5,50,95パーセンタイル点）を表示してください。

In [ ]:
df.groupby("Species_short", observed=True)["Body_Mass"].quantile([0.05, 0.5, 0.95])

全体のIQRからの外れ値とz値の絶対値が3超となる件数を表示してください。ただし、箱ひげ図のヒゲの長さはIQRの1.5倍とします。

In [ ]:
q1, q3 = np.quantile(x, [0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_mask = (x < lo) | (x > hi)
z = (x - m) / s
z_mask = np.abs(z) > 3
print(f"{lo=:.1f}, {hi=:.1f}, {int(iqr_mask.sum())=}")
print(f"{int(z_mask.sum())=}")

種ごとのIQRからの外れ値件数を表示してください。

In [ ]:
for sp, g in df.dropna(subset=["Body_Mass"]).groupby("Species_short", observed=True):
    vals = g["Body_Mass"].to_numpy()
    qq1, qq3 = np.quantile(vals, [0.25, 0.75])
    fences = (qq1 - 1.5 * (qq3 - qq1), qq3 + 1.5 * (qq3 - qq1))
    n_out = int(((vals < fences[0]) | (vals > fences[1])).sum())
    print(f"{sp=}, fences=[{fences[0]}, {fences[1]}], {n_out=}")